# Stabilization review

Confirm, post facto, that each channel's FSCV response stabilized. Reads whichever
stabilization files a session exported (`-start` / `-end` / `-full`), and per
**device-timepoint** plots: (1) raw FSCV I-V loops to pick `V_target`,
(2) `I(V_target)` over cycles, (3) the normalized cycle-drift convergence metric,
and (4) a convergence summary. Works on any root (in vitro or in vivo).

---

**Where this fits: the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review`: inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. `quality_filtering_dashboard`: set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. **`stabilization_review`: confirm each channel's FSCV response stabilized (optional / parallel)  ← you are here**
4. `diagnostics_review`: characterize the featureset before modeling (*is recalibration well-posed?*)
5. `discovery_checkpointed`: run the model-discovery task queue batch-by-batch under nested CV
6. `discovery_results_review`: inspect a finished discovery run; pick a model
7. `deployment_domain_shift`: freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup: run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    # --upgrade is load-bearing. `pip install electropycal` is a NO-OP when any version is
    # already present, and a hosted runtime often carries an older one from an earlier
    # session, so without it the cell reports success and leaves the old version in place.
    !pip install -q --upgrade electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive

# Check the version here, where the message is readable, instead of letting an older build
# fail deep inside a reader. If this raises immediately after an upgrade, restart the
# runtime: a module already imported stays in memory and the upgrade does not reach it.
import electropycal

_parts = electropycal.__version__.split(".")[:2]
if tuple(int(p) for p in _parts if p.isdigit()) < (0, 10):
    raise RuntimeError(
        f"electropycal {electropycal.__version__} is older than these notebooks need "
        f"(0.10 or newer, for the full range of export variants). Run "
        f"`pip install -q --upgrade electropycal`, then restart the runtime so the "
        f"upgraded version is the one imported.")

In [ ]:
%matplotlib inline
import tempfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from electropycal.viz import set_pub_style
from electropycal.stabreview import (StabilizationIndex, plot_raw_cycles, plot_i_vtarget,
    plot_round_drift, convergence_table)
set_pub_style()

# --- point ROOT at your data folder (Google Drive) ---
ROOT = None   # <- your own PSTrace export directory; None uses the shipped demo
# demo_input() locates demo/ from ANY working directory -- the repo root, notebooks/,
# or anywhere else -- and synthesizes an equivalent tree when no checkout is reachable
# (e.g. Colab). A relative literal here breaks in whichever cwd it was not written for.
# ROOT must be a LOCAL FILESYSTEM PATH. A Drive or cloud share URL will not work:
# nothing here downloads for you. On Colab, mount the drive first and use the
# mounted path:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     ROOT = '/content/drive/MyDrive/<your-folder>'
# Point ROOT at the directory that DIRECTLY CONTAINS the <YYYYMMDD>_..._signal
# folders, not at its parent. If you are one level off, extraction now says so and
# names the directory it thinks you meant.
from electropycal._demo import demo_input
ROOT = demo_input("in_vitro", ROOT)
print("ROOT =", ROOT)

## Available data: stabilization files by device / date / timepoint
`timepoint` = elapsed days since each device's first signal session (matching the
rest of the pipeline), shown as `D0`, `D1`, …. Use this to choose `SELECT` below.

In [ ]:
# Inventory of stabilization files + the settling parameters (v_target, tol, patience, ...).
IDX = StabilizationIndex(ROOT)
IDX.print_settings()
print()
av = IDX.available_table()
print(f"ROOT = {ROOT}  |  {len(av)} device-timepoint session(s):")
with pd.option_context("display.max_rows", None):
    display(IDX.label_devices(av))

## Analysis config
`DEVICE_TYPES` picks device types (`neurostring` / `cfme` / …); `SELECT` picks which
device-timepoints to analyze. Both default to *all available*, each gets its own plots
(titled `<device-type> <id> - ...`). Set `V_TARGET` after looking at the raw loops below.

In [ ]:
# Selection is local to this run; settling parameters come from the index (override by rebuilding,
# e.g. StabilizationIndex(ROOT, v_target=0.7, tol=0.02, region="anodic") - see settings print above).
DEVICE_TYPES = None                 # None = all; or ["neurostring"], ["cfme"]
SELECT       = [("2-3", 0)]         # None = all device-timepoints; or [("3-2", 0), ("3-2", 4)]
PAIRS = IDX.pairs(select=SELECT, device_types=DEVICE_TYPES)
print(f"{len(PAIRS)} device-timepoint(s): " + ", ".join(f"{IDX.title(d)} D{int(t)}" for d, t in PAIRS))

## 1. Raw FSCV cycles: pick `V_target`
Full anodic+cathodic I-V loop per channel (channel color = depth via a viridis gradient).
Only **two cycles** are drawn: the **last cycle of the first round** (faded) and the **last cycle
of the last round** (solid). Both are settled past the within-round capacitive charging, so the
comparison reflects genuine round-over-round drift rather than a cold-start transient (e.g. legend
`cycle 20` vs `cycle 200` at `CYCLES_PER_ROUND=20`). From the `full` file if present, otherwise the
first round of `start` and the last round of `end`. Set `V_TARGET` (dashed) to the oxidation-peak
potential; DA window shaded.

In [ ]:
plot_raw_cycles(IDX, PAIRS)

## 2. I(V_target) over cycles
Oxidation current at `V_target` vs stabilization cycle, one panel per channel
(channel color). Round-boundary jumps are visible in the raw trace; a settled
channel is flat by the end.

In [ ]:
plot_i_vtarget(IDX, PAIRS)

## 3. Round-averaged cycle drift (convergence metric)
Each round's cycles are averaged into one waveform (the cold-start scan 1 dropped when
`DROP_COLDSTART`), then `drift = ‖rₖ − rₖ₋₁‖ / ‖rₖ‖` is taken between consecutive
**round** means on the **anodic** sweep (`REGION`). Averaging 20 cycles collapses the
per-cycle jitter, cold-starts, and glitch cycles that made the raw per-cycle metric
spiky, so the curve tracks the electrode's genuine round-over-round settling. Smoothed
(`SMOOTH_WINDOW`, in rounds); the channel is **converged** once it stays ≤ `TOL`
(dashed) for `PATIENCE` rounds. (x-axis is the round-to-round step.)

In [ ]:
plot_round_drift(IDX, PAIRS)

## 4. Convergence summary (all analyzed channels)
From the richest available phase (`full` > `end` > `start`), on the round-averaged
`REGION` drift:
- **`settle_tau`**: τ (in **rounds**) of an `A·e^{−t/τ}+c` fit to the drift envelope (fast = small).
- **`final_drift`**: smoothed round-drift on the last round.
- **`n_stable`**: trailing run of **rounds** with smoothed drift ≤ `TOL`.
- **`converged`**: `n_stable ≥ PATIENCE` rounds.

(`settle_tau`/`converged` need several rounds; `-start`/`-end` captures with only a few
rounds will read `NaN`/`False`; use a `full` stabilization file to judge convergence.)

In [ ]:
summary_tbl = convergence_table(IDX, PAIRS)
print("converged:", int(summary_tbl.converged.sum()), "/", len(summary_tbl),
      f"channels (region={IDX.region}, tol={IDX.tol}, patience={IDX.patience})")
with pd.option_context("display.max_rows", None):
    display(IDX.label_devices(summary_tbl))